# Chuẩn bị Colab và Drive
Chọn CPU runtime. Clone mã nguồn GitHub hiện tại và ghi commit dùng cho thực nghiệm. Không cần ZIP source hoặc ZIP bản vá cũ. Đẩy source mới lên GitHub trước khi chạy notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile, time
from collections import deque

BASE = Path('/content/drive/MyDrive/BigData/DoAn_GoiYPhim_Alluxio')
PROJECT = Path('/content/movie-recommendation-alluxio')
REPOSITORY = 'https://github.com/iuzenaurora/movie-recommendation-alluxio.git'
SOURCE_REF = 'origin/main'  # Use a commit SHA for a reproducible experiment.
for name in ['01_Dataset_Raw','02_Dataset_Processed','03_Notebooks','04_Models','05_Experiments','06_Documents']:
    (BASE / name).mkdir(parents=True, exist_ok=True)
if not PROJECT.exists():
    subprocess.run(['git','clone',REPOSITORY,str(PROJECT)],check=True)
if not (PROJECT / '.git').is_dir():
    raise ValueError('PROJECT already exists without Git; select a new project directory.')
if subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip():
    raise ValueError('Source has local changes. Preserve them before selecting another revision.')
subprocess.run(['git','-C',str(PROJECT),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_REF],check=True)
os.chdir(PROJECT)
java_home = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not java_home.exists():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pyspark==3.5.6','numpy','matplotlib'],check=True)
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home / 'bin') + ':' + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
sys.path.insert(0,str(PROJECT / 'src'))
import numpy as np
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Python:',sys.version,'NumPy:',np.__version__)
subprocess.run(['java','-version'],check=True)

def extract_checked(path, destination):
    with zipfile.ZipFile(path) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt ZIP: ' + str(path))
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe ZIP path')
        archive.extractall(destination)


In [ ]:
subprocess.run(['java','-version'],check=True)
subprocess.run([sys.executable,'scripts/download.py','--dataset','small','--out',str(BASE/'01_Dataset_Raw')],check=True)

Sau khi quy trình small thành công, thay small bằng 32m để tải dữ liệu chính. Dung lượng Drive không đại diện dung lượng RAM hoặc đĩa Colab.